# Hy3 API - 流式请求示例

本示例演示如何使用 Hy3 API 的流式请求功能，实现逐 token 输出和实时响应。

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
client = OpenAI(
    api_key=os.getenv("API_KEY", "EMPTY"),
    base_url=os.getenv("BASE_URL", "http://127.0.0.1:8000/v1")
)


## 流式请求基础

In [ ]:
stream = client.chat.completions.create(
    model="hy3",
    messages=[
        {"role": "user", "content": "请用简短的语言介绍 Python 的主要特点"},
    ],
    temperature=0.9,
    top_p=1.0,
    stream=True,
)

full_content = ""
chunk_count = 0

for chunk in stream:
    chunk_count += 1
    if chunk.choices:
        delta = chunk.choices[0].delta
        if delta.content:
            full_content += delta.content
            print(f"[Chunk #{chunk_count}] '{delta.content}'")

print(f"\n完整回复:\n{full_content}")

## Chunk 结构解析

In [ ]:
stream = client.chat.completions.create(
    model="hy3",
    messages=[{"role": "user", "content": "你好"}],
    stream=True,
)

for chunk in stream:
    print(f"\n=== Chunk ===")
    print(f"id: {chunk.id}")
    print(f"object: {chunk.object}")
    print(f"created: {chunk.created}")
    print(f"model: {chunk.model}")

    if chunk.choices:
        choice = chunk.choices[0]
        delta = choice.delta
        print(f"\nchoices[0]:")
        print(f"  finish_reason: {choice.finish_reason}")
        print(f"  delta:")
        if delta.role:
            print(f"    ├─ role: {delta.role}")
        if delta.content:
            print(f"    └─ content: '{delta.content}'")

## 流式请求 + 工具调用

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取指定城市的天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "城市名称"},
                },
                "required": ["city"],
            },
        },
    }
]

stream = client.chat.completions.create(
    model="hy3",
    messages=[{"role": "user", "content": "北京今天天气怎么样？"}],
    tools=tools,
    stream=True,
)

for chunk in stream:
    if chunk.choices:
        delta = chunk.choices[0].delta
        if delta.tool_calls:
            for tc in delta.tool_calls:
                print(f"\n工具调用:")
                print(f"  name: {tc.function.name}")
                print(f"  arguments: {tc.function.arguments}")

## 关键要点

1. **实时输出**：流式请求可以在响应生成过程中逐字显示，提升用户体验
2. **首 Token 时延**：用户可以更快看到第一个字符，减少等待感
3. **内存效率**：不需要等待完整响应，可以边接收边处理
4. **finish_reason**：通过检查 `finish_reason` 判断响应是否结束